In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df =  pd.read_csv("train.csv")

In [ ]:
df.shape

In [ ]:
from datetime import datetime
df['date'] = pd.to_datetime(df['timestamp'], unit = 's')
df = df.sort_values('date')
groups = pd.factorize(df['date'].dt.day.astype(str) + '_' + df['date'].dt.month.astype(str) + '_' + df['date'].dt.year.astype(str))[0]

In [ ]:
btc = df[df["Asset_ID"]==1].set_index("date")

In [ ]:
import plotly.graph_objects as go
btc_mini = btc.iloc[-200:]
fig = go.Figure(data=[go.Candlestick(x=btc_mini.index, open=btc_mini['Open'], high=btc_mini['High'], low=btc_mini['Low'], close=btc_mini['Close'])])
fig.show()

In [ ]:
import pickle as pkl
pickle_out = open("btc_mini.pkl","wb")
pkl.dump(btc_mini,pickle_out)
pickle_out.close()

In [ ]:
btc.fillna(method="ffill",axis=0, inplace=True)

In [ ]:
btc.drop(columns=['timestamp','Asset_ID','Target','Volume'],axis=1,inplace=True)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))
btc[['Count', 'Open', 'High', 'Low', 'Close', 'VWAP']]=scaler.fit_transform(btc[['Count', 'Open', 'High', 'Low', 'Close', 'VWAP']])

In [ ]:
y = btc['Close']
X = btc.drop(columns=['Close'],axis=1)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=.20,random_state=0)

In [ ]:
X_train=np.array(X_train)
y_train=np.array(y_train)

In [ ]:
X_train=np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,Dropout

In [ ]:
X_train.shape

In [ ]:
model = Sequential()
model.add(LSTM(50,return_sequences=True,activation='relu',input_shape=(X_train.shape[1],1)))
model.add(LSTM(50,return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))
model.compile(loss='mean_squared_error',optimizer='adam')

In [ ]:
model.summary()

In [ ]:
model.fit(X_train,y_train,verbose=1,epochs=10,batch_size=128)

In [ ]:
losses=pd.DataFrame(model.history.history)
losses.plot()

In [ ]:
model.save("btc.h5")

In [ ]:
X_test = np.array(X_test)
X_test = np.reshape(X_test,(X_test.shape[0],X_test.shape[1],1))
y_test = np.array(y_test)

In [ ]:
X_test.shape

In [ ]:
y_test = y_test.reshape(391257,1)

In [ ]:
X_test[0].shape

In [ ]:
pred=model.predict(X_test)
#pred=scaler.inverse_transform(pred)
pred

In [ ]:
pred.shape

In [ ]:
rmse=np.sqrt(np.mean(((pred-y_test)**2)))
rmse